# Phase 1-1: CAUEEG 데이터셋 개요 및 기본 통계

**목표**: CAUEEG 데이터셋의 구조 이해 및 품질 확인

**분석 내용**:
1. 데이터 로딩 및 기본 구조 확인
2. Class별 나이 분포 (confounding factor 확인)
3. Class별 녹음 길이 분포
4. 신호 품질 확인 (채널별 진폭, 결측값)
5. Train/Val/Test split 통계 요약

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from collections import Counter

# 프로젝트 루트를 path에 추가
sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

DATASET_PATH = '../local/dataset/caueeg-dataset'
SAMPLING_RATE = 200  # Hz

## 1. 데이터 로딩

기존 CeedNet 코드의 `load_caueeg_task_datasets()`를 재사용하여 CAUEEG-Dementia 태스크를 로딩합니다.

In [ ]:
from datasets.caueeg_script import load_caueeg_task_datasets, load_caueeg_config

# 데이터셋 설정 로딩
dataset_config = load_caueeg_config(DATASET_PATH)
print("Dataset config keys:", list(dataset_config.keys()))
print("Signal header:", dataset_config['signal_header'])

# CAUEEG-Dementia 태스크 로딩 (이벤트 포함)
task_config, train_ds, val_ds, test_ds = load_caueeg_task_datasets(
    DATASET_PATH, task='dementia', load_event=True, file_format='edf'
)

print(f"\nClass labels: {task_config['class_label_to_name']}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# 샘플 데이터 구조 확인
sample = train_ds[0]
print("Sample keys:", list(sample.keys()))
print(f"Serial: {sample['serial']}")
print(f"Age: {sample['age']}")
print(f"Class label: {sample['class_label']} ({task_config['class_label_to_name'][sample['class_label']]})")
print(f"Signal shape: {sample['signal'].shape}  (channels x samples)")
print(f"Recording length: {sample['signal'].shape[1] / SAMPLING_RATE:.1f} seconds")
print(f"Number of events: {len(sample['event'])}")
print(f"First 5 events: {sample['event'][:5]}")

## 2. 전체 데이터 통계 수집

모든 split의 메타데이터를 수집하여 DataFrame으로 정리합니다.

In [ ]:
def collect_metadata(dataset, split_name, class_names):
    """데이터셋에서 메타데이터를 수집하여 DataFrame 행 리스트로 반환"""
    rows = []
    for i in range(len(dataset)):
        sample = dataset[i]
        signal = sample['signal']
        n_samples = signal.shape[1]
        
        rows.append({
            'serial': sample['serial'],
            'split': split_name,
            'age': sample['age'],
            'class_label': sample['class_label'],
            'class_name': class_names[sample['class_label']],
            'n_samples': n_samples,
            'duration_sec': n_samples / SAMPLING_RATE,
            'duration_min': n_samples / SAMPLING_RATE / 60,
            'n_events': len(sample.get('event', [])),
            'has_nan': np.any(np.isnan(signal)),
            'signal_min': np.min(signal[:19]),  # EEG channels only
            'signal_max': np.max(signal[:19]),
            'signal_std': np.std(signal[:19]),
        })
        
        if (i + 1) % 100 == 0:
            print(f"  {split_name}: {i+1}/{len(dataset)} processed")
    
    return rows

class_names = task_config['class_label_to_name']

print("Collecting metadata from all splits...")
all_rows = []
for ds, name in [(train_ds, 'train'), (val_ds, 'val'), (test_ds, 'test')]:
    all_rows.extend(collect_metadata(ds, name, class_names))
    print(f"  {name}: done ({len(ds)} samples)")

df = pd.DataFrame(all_rows)
print(f"\nTotal samples: {len(df)}")
df.head()

## 3. Split별 Class 분포

In [ ]:
# Split별 class 분포 테이블
split_class_counts = df.groupby(['split', 'class_name']).size().unstack(fill_value=0)
split_class_counts = split_class_counts[['Normal', 'MCI', 'Dementia']]
split_class_counts['Total'] = split_class_counts.sum(axis=1)
print("Split별 Class 분포:")
print(split_class_counts)
print()

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, split in enumerate(['train', 'val', 'test']):
    split_df = df[df['split'] == split]
    counts = split_df['class_name'].value_counts()[['Normal', 'MCI', 'Dementia']]
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    axes[i].bar(counts.index, counts.values, color=colors)
    axes[i].set_title(f'{split.upper()} (n={len(split_df)})')
    axes[i].set_ylabel('Count')
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 1, str(v), ha='center', fontweight='bold')

plt.suptitle('Class Distribution by Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Class별 나이 분포 (Confounding Factor 확인)

나이가 class와 강하게 상관되어 있다면 confounding factor로 작용할 수 있습니다.
EDCC 모델에서 Age-conditioned normalization과 Age adversarial training이 이를 해결하기 위한 설계입니다.

In [ ]:
# Class별 나이 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram + KDE
colors = {'Normal': '#2ecc71', 'MCI': '#f39c12', 'Dementia': '#e74c3c'}
for cls in ['Normal', 'MCI', 'Dementia']:
    subset = df[df['class_name'] == cls]['age']
    axes[0].hist(subset, bins=20, alpha=0.5, label=f'{cls} (n={len(subset)}, mean={subset.mean():.1f})', 
                 color=colors[cls], density=True)
    subset.plot.kde(ax=axes[0], color=colors[cls], linewidth=2)

axes[0].set_xlabel('Age')
axes[0].set_ylabel('Density')
axes[0].set_title('Age Distribution by Class')
axes[0].legend()

# Box plot
class_order = ['Normal', 'MCI', 'Dementia']
df_plot = df[df['class_name'].isin(class_order)]
sns.boxplot(data=df_plot, x='class_name', y='age', order=class_order, 
            palette=colors, ax=axes[1])
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Age')
axes[1].set_title('Age Distribution by Class (Boxplot)')

plt.tight_layout()
plt.show()

# 통계 요약
print("\nClass별 나이 통계:")
print(df.groupby('class_name')['age'].describe()[['count', 'mean', 'std', 'min', '50%', 'max']])

In [ ]:
# Kruskal-Wallis 검정 (나이의 class 간 차이)
normal_ages = df[df['class_name'] == 'Normal']['age']
mci_ages = df[df['class_name'] == 'MCI']['age']
dementia_ages = df[df['class_name'] == 'Dementia']['age']

stat, p_value = stats.kruskal(normal_ages, mci_ages, dementia_ages)
print(f"Kruskal-Wallis test: H={stat:.4f}, p={p_value:.2e}")

if p_value < 0.05:
    print("=> 나이가 class 간에 유의미하게 다릅니다 (p < 0.05)")
    print("   → Age-conditioned normalization과 Age adversarial training이 필요합니다.")
    
    # 사후 검정: Mann-Whitney U (pairwise)
    pairs = [('Normal', 'MCI'), ('Normal', 'Dementia'), ('MCI', 'Dementia')]
    print("\nPairwise Mann-Whitney U tests:")
    for c1, c2 in pairs:
        a1 = df[df['class_name'] == c1]['age']
        a2 = df[df['class_name'] == c2]['age']
        u_stat, u_p = stats.mannwhitneyu(a1, a2, alternative='two-sided')
        print(f"  {c1} vs {c2}: U={u_stat:.0f}, p={u_p:.2e} {'*' if u_p < 0.05/3 else ''}")
else:
    print("=> 나이가 class 간에 유의미한 차이가 없습니다.")

## 5. Class별 녹음 길이 분포

In [ ]:
# Class별 녹음 길이 분포
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in ['Normal', 'MCI', 'Dementia']:
    subset = df[df['class_name'] == cls]['duration_min']
    axes[0].hist(subset, bins=30, alpha=0.5, label=f'{cls} ({subset.mean():.1f} min)', 
                 color=colors[cls])

axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Recording Duration by Class')
axes[0].legend()

sns.boxplot(data=df, x='class_name', y='duration_min', order=class_order, 
            palette=colors, ax=axes[1])
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Duration (minutes)')
axes[1].set_title('Recording Duration by Class (Boxplot)')

plt.tight_layout()
plt.show()

# 통계 요약
print("녹음 길이 통계 (분):")
print(df.groupby('class_name')['duration_min'].describe()[['count', 'mean', 'std', 'min', '50%', 'max']])

# EDCC 윈도우 수 추정
print(f"\n=== EDCC 윈도우 수 추정 (2초 윈도우, 1초 stride) ===")
df['est_windows'] = ((df['n_samples'] - 400) / 200 + 1).astype(int)
print(df.groupby('class_name')['est_windows'].describe()[['mean', 'std', 'min', 'max']])

## 6. 신호 품질 확인

In [ ]:
# NaN 확인
nan_count = df['has_nan'].sum()
print(f"NaN이 포함된 녹음 수: {nan_count} / {len(df)}")

# 신호 진폭 범위
print(f"\nEEG 신호 진폭 통계:")
print(f"  전체 min: {df['signal_min'].min():.2f}")
print(f"  전체 max: {df['signal_max'].max():.2f}")
print(f"  평균 std: {df['signal_std'].mean():.2f}")

# 극단적 진폭 녹음 탐지
extreme = df[(df['signal_max'] > 5000) | (df['signal_min'] < -5000)]
print(f"\n극단적 진폭 녹음 (|amplitude| > 5000): {len(extreme)}개")
if len(extreme) > 0:
    print(extreme[['serial', 'class_name', 'signal_min', 'signal_max']].to_string())

In [ ]:
# 채널별 신호 통계 (랜덤 100개 샘플에서 계산)
np.random.seed(42)
sample_indices = np.random.choice(len(train_ds), min(100, len(train_ds)), replace=False)

channel_names = [ch.replace('-AVG', '') for ch in dataset_config['signal_header']]
channel_stats = {ch: {'means': [], 'stds': []} for ch in channel_names[:19]}

for idx in sample_indices:
    sample = train_ds[idx]
    signal = sample['signal'][:19]  # EEG only
    for c in range(19):
        channel_stats[channel_names[c]]['means'].append(np.mean(signal[c]))
        channel_stats[channel_names[c]]['stds'].append(np.std(signal[c]))

# 채널별 평균 std 시각화
ch_names = list(channel_stats.keys())
ch_mean_stds = [np.mean(channel_stats[ch]['stds']) for ch in ch_names]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(ch_names)), ch_mean_stds, color='steelblue')
ax.set_xticks(range(len(ch_names)))
ax.set_xticklabels(ch_names, rotation=45)
ax.set_ylabel('Mean STD')
ax.set_title('Channel-wise Signal Standard Deviation (100 random training samples)')

# 뇌 영역 색상 구분
region_colors = {
    'Frontal': '#e74c3c', 'Central': '#3498db', 'Parietal': '#2ecc71',
    'Temporal': '#9b59b6', 'Occipital': '#f39c12'
}
region_map = {
    'Fp1': 'Frontal', 'F3': 'Frontal', 'C3': 'Central', 'P3': 'Parietal', 'O1': 'Occipital',
    'Fp2': 'Frontal', 'F4': 'Frontal', 'C4': 'Central', 'P4': 'Parietal', 'O2': 'Occipital',
    'F7': 'Frontal', 'T3': 'Temporal', 'T5': 'Temporal', 'F8': 'Frontal', 'T4': 'Temporal',
    'T6': 'Temporal', 'FZ': 'Frontal', 'CZ': 'Central', 'PZ': 'Parietal'
}
for i, ch in enumerate(ch_names):
    bars[i].set_color(region_colors[region_map[ch]])

# 범례
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in region_colors.items()]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 7. 샘플 EEG 신호 시각화

각 class별로 샘플 녹음의 EEG 신호를 시각화하여 직관적으로 차이를 확인합니다.

In [ ]:
# 각 class에서 첫 번째 샘플의 10초 구간 시각화
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# class별 첫 번째 샘플 찾기
for ax_idx, target_class in enumerate([0, 1, 2]):
    for i in range(len(train_ds)):
        sample = train_ds[i]
        if sample['class_label'] == target_class:
            signal = sample['signal'][:19]  # EEG channels only
            # 녹음 중간 부분에서 10초 (2000 samples) 추출
            mid = signal.shape[1] // 2
            start = max(0, mid - 1000)
            segment = signal[:, start:start + 2000]
            
            time_axis = np.arange(segment.shape[1]) / SAMPLING_RATE
            
            # 채널별로 offset을 주어 시각화
            for ch in range(19):
                offset = ch * 100  # uV offset
                axes[ax_idx].plot(time_axis, segment[ch] + offset, 
                                  linewidth=0.5, color='k', alpha=0.7)
            
            axes[ax_idx].set_yticks([ch * 100 for ch in range(19)])
            axes[ax_idx].set_yticklabels(channel_names[:19], fontsize=8)
            axes[ax_idx].set_title(
                f"{class_names[target_class]} (serial={sample['serial']}, age={sample['age']})",
                fontweight='bold', color=list(colors.values())[ax_idx]
            )
            axes[ax_idx].set_xlabel('Time (seconds)')
            break

plt.suptitle('Sample EEG Signals (10-second segments)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. 요약

**주요 확인 사항 체크리스트**:
- [ ] 나이가 class 간에 유의미하게 다른지 → Age conditioning 필요성 확인
- [ ] 녹음 길이가 class 간에 유의미하게 다른지 → 편향 확인
- [ ] NaN/결측값 존재 여부
- [ ] 극단적 진폭 녹음 존재 여부 → 전처리 시 클리핑 필요 여부
- [ ] 채널별 신호 특성 확인 → 영역별 차이 존재 여부

**다음 단계**: Phase 1-2에서 이벤트 구조를 분석하여 EDCC 모델의 Event-Aware Tokenization 전략을 수립합니다.